# Smart Ride AI
Road analysis · vibration simulation · adaptive suspension · Streamlit dashboard

Run cells top to bottom. All outputs save to `smart_ride_ai/` folder.

---

In [ ]:
# CELL 1 — install deps
!pip install opencv-python-headless numpy pandas matplotlib plotly scikit-learn streamlit scipy joblib --quiet
print('done')


In [ ]:
# CELL 2 — create project folders
import os

dirs = [
    'smart_ride_ai/road_analysis',
    'smart_ride_ai/vibration_engine',
    'smart_ride_ai/suspension_sim',
    'smart_ride_ai/alert_system',
    'smart_ride_ai/road_memory',
    'smart_ride_ai/dashboard',
    'smart_ride_ai/data',
    'smart_ride_ai/models',
    'smart_ride_ai/assets',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    init = os.path.join(d, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

print('folders ready')


---
## Phase 1 — Synthetic road video

In [ ]:
# CELL 3 — generate road video
import cv2
import numpy as np

def make_road_video(out='smart_ride_ai/data/road_test.avi', nframes=300, w=640, h=480):
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    writer = cv2.VideoWriter(out, fourcc, 30.0, (w, h))
    rng = np.random.default_rng(42)

    for i in range(nframes):
        frame = np.full((h, w, 3), 58, dtype=np.uint8)
        noise = rng.integers(0, 22, (h, w, 3), dtype=np.uint8)
        frame = cv2.add(frame, noise)

        # scrolling lane dash
        offset = (i * 8) % 80
        for y in range(-offset, h, 80):
            if 0 <= y < h - 30:
                cv2.rectangle(frame, (308, y), (332, y + 40), (175, 175, 95), -1)

        # pothole ellipse — new one every 30 frames
        if i % 30 < 20:
            ph_y = ((i % 30) * 16 + 50) % h
            ph_x = 180 + (i // 30) % 4 * 90
            cv2.ellipse(frame, (ph_x, ph_y), (34, 19), 0, 0, 360, (12, 10, 8), -1)
            cv2.ellipse(frame, (ph_x, ph_y), (34, 19), 0, 0, 360, (42, 38, 32), 2)

        # cracks every 15 frames
        if i % 15 == 0:
            rng2 = np.random.default_rng(i)
            for _ in range(3):
                x1 = int(rng2.integers(50, w - 50))
                y1 = int(rng2.integers(50, h - 50))
                x2 = int(x1 + rng2.integers(-55, 55))
                y2 = int(y1 + rng2.integers(-55, 55))
                cv2.line(frame, (x1, y1), (x2, y2), (18, 15, 12), 1)

        # speed bump
        if 100 <= i <= 116 or 200 <= i <= 216:
            cv2.rectangle(frame, (0, 236), (w, 252), (88, 78, 58), -1)

        writer.write(frame)

    writer.release()
    print(f'video saved to {out}')

make_road_video()


In [ ]:
# CELL 4 — preview 5 frames
import cv2
import matplotlib.pyplot as plt

cap   = cv2.VideoCapture('smart_ride_ai/data/road_test.avi')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
idxs  = [int(i * total / 5) for i in range(5)]

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
fig.patch.set_facecolor('#0d0d0d')

for ax, idx in zip(axes, idxs):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, f = cap.read()
    if ret:
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        ax.set_title(f'f{idx}', color='#aaa', fontsize=8)
        ax.axis('off')

cap.release()
plt.suptitle('road feed preview', color='#00ffcc', fontsize=11)
plt.tight_layout()
plt.show()


---
## Phase 2 — Road surface analysis

In [ ]:
# CELL 5 — write road_analysis/analyzer.py
src = '''
import cv2
import numpy as np


class RoadAnalyzer:

    POTHOLE_THRESH = 45
    BUMP_THRESH    = 110
    MIN_PH_AREA    = 400
    MIN_BUMP_WIDTH = 200

    def analyze(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape

        # pothole — dark blob detection
        _, dark = cv2.threshold(gray, self.POTHOLE_THRESH, 255, cv2.THRESH_BINARY_INV)
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, k)
        cnts, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        potholes = [c for c in cnts if cv2.contourArea(c) > self.MIN_PH_AREA]

        # bumps — bright wide regions
        _, bright = cv2.threshold(gray, self.BUMP_THRESH, 255, cv2.THRESH_BINARY)
        bcnts, _ = cv2.findContours(bright, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        bumps = []
        for c in bcnts:
            x, y, bw, bh = cv2.boundingRect(c)
            if bw > self.MIN_BUMP_WIDTH and bw > bh * 3:
                bumps.append(c)

        # cracks via edge detector
        edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 30, 90)
        crack_px = int(np.sum(edges > 0))

        # roughness — empirical blend
        std    = float(np.std(gray))
        dark_r = float(np.sum(dark > 0)) / (h * w)
        roughness = min(100, int(std * 0.8 + dark_r * 200 + len(potholes) * 15))

        return {
            "potholes"     : potholes,
            "bumps"        : bumps,
            "edges"        : edges,
            "pothole_count": len(potholes),
            "bump_count"   : len(bumps),
            "crack_px"     : crack_px,
            "roughness"    : roughness,
            "smoothness"   : max(0, 100 - roughness),
            "std"          : round(std, 2),
            "dark_ratio"   : round(dark_r, 4),
        }

    def road_class(self, r):
        s, p = r["roughness"], r["pothole_count"]
        if p >= 3 or s >= 80: return "CRITICAL", "#ff2222"
        if p >= 1 or s >= 55: return "WARNING",  "#ff9900"
        if s >= 30:            return "CAUTION",  "#ffdd00"
        return                        "GOOD",     "#00ff88"
'''

with open('smart_ride_ai/road_analysis/analyzer.py', 'w') as f:
    f.write(src)
print('analyzer.py written')


In [ ]:
# CELL 6 — run analyzer on every frame
import sys, os
sys.path.insert(0, '.')

import cv2
import pandas as pd
from smart_ride_ai.road_analysis.analyzer import RoadAnalyzer

cap   = cv2.VideoCapture('smart_ride_ai/data/road_test.avi')
fps   = cap.get(cv2.CAP_PROP_FPS)
az    = RoadAnalyzer()
rows  = []
i     = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    r = az.analyze(frame)
    lbl, _ = az.road_class(r)
    rows.append({
        'frame'        : i,
        'ts'           : round(i / fps, 2),
        'roughness'    : r['roughness'],
        'smoothness'   : r['smoothness'],
        'pothole_count': r['pothole_count'],
        'bump_count'   : r['bump_count'],
        'crack_px'     : r['crack_px'],
        'road_class'   : lbl,
    })
    i += 1

cap.release()
df_road = pd.DataFrame(rows)
df_road.to_csv('smart_ride_ai/data/road_analysis.csv', index=False)
print(f'{len(df_road)} frames done')
print(df_road[['frame', 'roughness', 'pothole_count', 'road_class']].head(8).to_string())


In [ ]:
# CELL 7 — road condition timeline chart
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('smart_ride_ai/data/road_analysis.csv')

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.patch.set_facecolor('#0a0a0f')

cfg = [
    ('roughness',     '#ff4466', 'roughness'),
    ('pothole_count', '#ff9900', 'potholes/frame'),
    ('smoothness',    '#00ffcc', 'smoothness'),
]

for ax, (col, clr, lbl) in zip(axes, cfg):
    ax.set_facecolor('#111118')
    ax.plot(df['ts'], df[col], color=clr, lw=0.9)
    ax.fill_between(df['ts'], df[col], alpha=0.12, color=clr)
    ax.set_ylabel(lbl, color='#aaa', fontsize=9)
    ax.tick_params(colors='#555')
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#222')

axes[0].axhline(55, color='#ff9900', ls='--', lw=0.7, alpha=0.5)
axes[0].axhline(80, color='#ff2222', ls='--', lw=0.7, alpha=0.5)
axes[2].set_xlabel('time (s)', color='#aaa')
plt.suptitle('road condition timeline', color='#00ffcc', fontsize=12)
plt.tight_layout()
plt.savefig('smart_ride_ai/assets/road_timeline.png', dpi=140, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()


---
## Phase 3 — Vibration and motion analysis

In [ ]:
# CELL 8 — write vibration_engine/vibration.py
src = '''
import numpy as np


class VibrationEngine:

    def simulate(self, roughness, frame_idx):
        rng = np.random.default_rng(frame_idx)
        r   = roughness / 100.0

        freq_z = 1.5 + r * 4.0
        amp_z  = 0.2 + r * 2.8
        phase  = frame_idx / 30.0 * freq_z * 2 * np.pi
        z = amp_z * np.sin(phase) + float(rng.normal(0, r * 0.4 + 1e-9))

        if r > 0.7 and rng.random() < 0.15:
            z += float(rng.choice([-1, 1])) * float(rng.uniform(2.0, 4.5))

        x = (0.05 + r * 0.6) * np.sin(phase * 0.7 + 1.2) + float(rng.normal(0, r * 0.15 + 1e-9))
        y = (0.05 + r * 0.4) * np.cos(phase * 0.5 + 0.8) + float(rng.normal(0, r * 0.1 + 1e-9))

        rms = float(np.sqrt(x**2 + y**2 + z**2))
        stability = max(0, 100 - int(rms * 22))

        return {
            "ax": round(x, 4),
            "ay": round(y, 4),
            "az": round(z, 4),
            "rms": round(rms, 4),
            "stability": stability,
        }

    def iso_class(self, rms):
        if rms < 0.315: return "not uncomfortable"
        if rms < 0.630: return "a little uncomfortable"
        if rms < 1.000: return "fairly uncomfortable"
        if rms < 1.600: return "uncomfortable"
        if rms < 2.500: return "very uncomfortable"
        return "extremely uncomfortable"
'''

with open('smart_ride_ai/vibration_engine/vibration.py', 'w') as f:
    f.write(src)
print('vibration.py written')


In [ ]:
# CELL 9 — run vibration simulation
import sys
sys.path.insert(0, '.')
import pandas as pd
from smart_ride_ai.vibration_engine.vibration import VibrationEngine

df   = pd.read_csv('smart_ride_ai/data/road_analysis.csv')
eng  = VibrationEngine()
rows = []

for _, row in df.iterrows():
    v = eng.simulate(row['roughness'], int(row['frame']))
    rows.append(v)

df_out = pd.concat([df, pd.DataFrame(rows)], axis=1)
df_out.to_csv('smart_ride_ai/data/vibration_data.csv', index=False)
print(f'{len(df_out)} rows saved')
print(df_out[['ts', 'roughness', 'az', 'rms', 'stability']].head(8).to_string())


In [ ]:
# CELL 10 — 3-axis accel plot with ISO 2631-1 thresholds
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('smart_ride_ai/data/vibration_data.csv')
t  = df['ts']

fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
fig.patch.set_facecolor('#0a0a0f')

panels = [
    ('ax',  '#00aaff', 'lateral X  m/s2'),
    ('ay',  '#ffaa00', 'longitudinal Y  m/s2'),
    ('az',  '#ff3366', 'vertical Z  m/s2'),
    ('rms', '#00ffcc', 'RMS  m/s2'),
]

for ax, (col, clr, lbl) in zip(axes, panels):
    ax.set_facecolor('#111118')
    ax.plot(t, df[col], color=clr, lw=0.7, alpha=0.9)
    ax.fill_between(t, df[col], alpha=0.1, color=clr)
    ax.set_ylabel(lbl, color='#aaa', fontsize=8)
    ax.tick_params(colors='#555', labelsize=7)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#222')

for thresh, clr in [(0.315, '#00ff88'), (0.630, '#ffdd00'), (1.0, '#ff9900'), (1.6, '#ff2222')]:
    axes[3].axhline(thresh, color=clr, ls=':', lw=0.7, alpha=0.6)

axes[3].set_xlabel('time (s)', color='#aaa')
plt.suptitle('3-axis vibration  |  ISO 2631-1 comfort bands on RMS', color='#00ffcc', fontsize=11)
plt.tight_layout()
plt.savefig('smart_ride_ai/assets/vibration_plot.png', dpi=140, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()


---
## Phase 4 — AI comfort prediction

In [ ]:
# CELL 11 — train Random Forest comfort model
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score
import joblib

df = pd.read_csv('smart_ride_ai/data/vibration_data.csv')

FEATS = ['roughness', 'rms', 'az', 'pothole_count', 'crack_px', 'bump_count', 'stability']

def comfort_score(row):
    s  = 100
    s -= row['roughness'] * 0.40
    s -= min(40, row['rms'] * 15)
    s -= row['pothole_count'] * 8
    s -= min(10, row['crack_px'] / 500)
    return float(np.clip(s, 0, 100))

def comfort_class(score):
    if score >= 75: return 0
    if score >= 50: return 1
    if score >= 25: return 2
    return 3

df['comfort']       = df.apply(comfort_score, axis=1)
df['comfort_class'] = df['comfort'].apply(comfort_class)
df.to_csv('smart_ride_ai/data/labeled_data.csv', index=False)

X  = df[FEATS].fillna(0)
ys = df['comfort']
yc = df['comfort_class']
X_tr, X_te, ys_tr, ys_te, yc_tr, yc_te = train_test_split(X, ys, yc, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
reg.fit(X_tr, ys_tr)
print(f'regressor MAE : {mean_absolute_error(ys_te, reg.predict(X_te)):.2f}')

clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf.fit(X_tr, yc_tr)
print(f'classifier acc: {accuracy_score(yc_te, clf.predict(X_te)) * 100:.1f}%')

joblib.dump(reg, 'smart_ride_ai/models/comfort_reg.pkl')
joblib.dump(clf, 'smart_ride_ai/models/comfort_clf.pkl')

print()
print('feature importances:')
for f, imp in sorted(zip(FEATS, reg.feature_importances_), key=lambda x: -x[1]):
    print(f'  {f:<18} {chr(9608) * int(imp * 40)}  {imp:.3f}')


In [ ]:
# CELL 12 — score all frames and plot
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

df  = pd.read_csv('smart_ride_ai/data/labeled_data.csv')
reg = joblib.load('smart_ride_ai/models/comfort_reg.pkl')
clf = joblib.load('smart_ride_ai/models/comfort_clf.pkl')

FEATS = ['roughness', 'rms', 'az', 'pothole_count', 'crack_px', 'bump_count', 'stability']
X = df[FEATS].fillna(0)

df['pred_comfort'] = reg.predict(X).clip(0, 100)
df['pred_class']   = clf.predict(X)

LABELS = {0: 'comfortable', 1: 'mild discomfort', 2: 'high discomfort', 3: 'critical'}
COLORS = {0: '#00ff88',     1: '#ffdd00',          2: '#ff9900',         3: '#ff2222'}
df['comfort_label'] = df['pred_class'].map(LABELS)
df.to_csv('smart_ride_ai/data/comfort_scores.csv', index=False)

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.patch.set_facecolor('#0a0a0f')
for ax in axes:
    ax.set_facecolor('#111118')
    ax.tick_params(colors='#555')
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#222')

axes[0].plot(df['ts'], df['pred_comfort'], color='#00ffcc', lw=1.0)
axes[0].fill_between(df['ts'], df['pred_comfort'], alpha=0.12, color='#00ffcc')
for thresh, clr in [(75, '#00ff88'), (50, '#ffdd00'), (25, '#ff2222')]:
    axes[0].axhline(thresh, color=clr, ls='--', lw=0.6, alpha=0.45)
axes[0].set_ylabel('comfort score', color='#aaa')
axes[0].set_ylim(0, 105)
axes[0].set_title('predicted ride comfort', color='#00ffcc', fontsize=10)

counts = df['pred_class'].value_counts().sort_index()
bars   = axes[1].bar([LABELS[i] for i in counts.index], counts.values,
                      color=[COLORS[i] for i in counts.index], alpha=0.82)
for b, v in zip(bars, counts.values):
    axes[1].text(b.get_x() + b.get_width() / 2, v + 1, str(v),
                 ha='center', color='#aaa', fontsize=8)
axes[1].set_ylabel('frames', color='#aaa')
axes[1].set_title('comfort class distribution', color='#00ffcc', fontsize=10)

plt.suptitle('Phase 4 — AI comfort prediction', color='#00ffcc', fontsize=11)
plt.tight_layout()
plt.savefig('smart_ride_ai/assets/comfort_pred.png', dpi=140, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print(f'mean comfort : {df["pred_comfort"].mean():.1f}')
print(f'worst frame  : {df["pred_comfort"].min():.1f}')


---
## Phase 5 — Adaptive suspension simulation

In [ ]:
# CELL 13 — write suspension_sim/suspension.py
src = '''
import numpy as np


class SuspensionSim:

    K_MIN, K_MAX = 15000, 55000
    C_MIN, C_MAX = 1500,  8000
    K_BASE       = 25000

    def compute(self, roughness, rms, pothole_count, pred_comfort):
        r = roughness / 100.0
        v = min(rms / 3.0, 1.0)

        k = self.K_BASE + (r * 0.3 - v * 0.2) * (self.K_MAX - self.K_BASE)
        k = float(np.clip(k, self.K_MIN, self.K_MAX))

        c = self.C_MIN + r * (self.C_MAX - self.C_MIN)
        if pothole_count > 0:
            c = min(self.C_MAX, c * 1.3)
        c = float(c)

        ride_h   = 160 - int(r * 30)
        passive  = max(0.0, float(100 - roughness * 0.6))
        adaptive = float(min(100.0, pred_comfort + (k / self.K_BASE - 1) * 5))
        gain     = max(0.0, adaptive - passive)

        return {
            "stiffness" : round(k),
            "damping"   : round(c),
            "ride_h_mm" : ride_h,
            "passive_c" : round(passive, 1),
            "adaptive_c": round(adaptive, 1),
            "gain"      : round(gain, 1),
            "mode"      : self._mode(k, c),
        }

    def _mode(self, k, c):
        if k < 20000 and c < 3000: return "COMFORT"
        if k > 45000 or c > 6500:  return "SPORT"
        if c > 5000:                return "ADAPTIVE_ROUGH"
        return                              "ADAPTIVE_NORMAL"
'''

with open('smart_ride_ai/suspension_sim/suspension.py', 'w') as f:
    f.write(src)
print('suspension.py written')


In [ ]:
# CELL 14 — run suspension sim
import sys
sys.path.insert(0, '.')
import pandas as pd
import matplotlib.pyplot as plt
from smart_ride_ai.suspension_sim.suspension import SuspensionSim

df  = pd.read_csv('smart_ride_ai/data/comfort_scores.csv')
sim = SuspensionSim()
rows = []

for _, row in df.iterrows():
    s = sim.compute(row['roughness'], row['rms'], row['pothole_count'], row['pred_comfort'])
    rows.append(s)

df_all = pd.concat([df, pd.DataFrame(rows)], axis=1)
df_all.to_csv('smart_ride_ai/data/suspension_data.csv', index=False)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
fig.patch.set_facecolor('#0a0a0f')
t = df_all['ts']

for ax in axes:
    ax.set_facecolor('#111118')
    ax.tick_params(colors='#555')
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#222')

axes[0].plot(t, df_all['adaptive_c'], color='#00ffcc', lw=1.2, label='adaptive')
axes[0].plot(t, df_all['passive_c'],  color='#ff4466', lw=1.0, ls='--', alpha=0.7, label='passive')
axes[0].fill_between(t, df_all['passive_c'], df_all['adaptive_c'], alpha=0.1, color='#00ffcc')
axes[0].set_ylabel('comfort', color='#aaa')
axes[0].set_title('adaptive vs passive', color='#00ffcc', fontsize=10)
axes[0].legend(facecolor='#1a1a2e', labelcolor='#aaa', fontsize=8)
axes[0].set_ylim(0, 105)

axes[1].plot(t, df_all['damping'], color='#ff9900', lw=1.0)
axes[1].fill_between(t, df_all['damping'], alpha=0.1, color='#ff9900')
axes[1].set_ylabel('damping Ns/m', color='#aaa')
axes[1].set_title('damping force', color='#00ffcc', fontsize=10)

mc   = df_all['mode'].value_counts()
mclr = {'COMFORT': '#00ff88', 'ADAPTIVE_NORMAL': '#00aaff', 'ADAPTIVE_ROUGH': '#ff9900', 'SPORT': '#ff3366'}
bars = axes[2].barh(mc.index, mc.values, color=[mclr.get(m, '#888') for m in mc.index], alpha=0.82)
for b, v in zip(bars, mc.values):
    axes[2].text(v + 0.5, b.get_y() + b.get_height() / 2, str(v), va='center', color='#aaa', fontsize=8)
axes[2].set_xlabel('frames', color='#aaa')
axes[2].set_title('suspension mode distribution', color='#00ffcc', fontsize=10)

plt.suptitle('Phase 5 — adaptive suspension', color='#00ffcc', fontsize=11)
plt.tight_layout()
plt.savefig('smart_ride_ai/assets/suspension.png', dpi=140, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print(f'avg comfort gain: +{df_all["gain"].mean():.1f} pts')


---
## Phase 6 — Road memory and alert engine

In [ ]:
# CELL 15 — road memory database
import pandas as pd
import json
import os
from datetime import datetime

class RoadMemoryDB:

    def __init__(self, path='smart_ride_ai/data/road_memory.json'):
        self.path = path
        self.data = self._load()

    def _load(self):
        if os.path.exists(self.path):
            with open(self.path) as f:
                return json.load(f)
        return {'segments': []}

    def _save(self):
        with open(self.path, 'w') as f:
            json.dump(self.data, f, indent=2)

    def add(self, ts, roughness, potholes, comfort):
        if roughness < 50 and potholes == 0:
            return
        seg = {
            'id'       : len(self.data['segments']) + 1,
            'ts'       : round(float(ts), 2),
            'lat'      : round(17.385 + float(ts) * 0.0001, 6),
            'lon'      : round(78.486 + float(ts) * 0.0001, 6),
            'roughness': int(roughness),
            'potholes' : int(potholes),
            'comfort'  : round(float(comfort), 1),
            'severity' : 'CRITICAL' if roughness > 80 else 'WARNING' if roughness > 55 else 'CAUTION',
            'at'       : datetime.now().strftime('%Y-%m-%d %H:%M'),
        }
        self.data['segments'].append(seg)
        self._save()

    def stats(self):
        s = self.data['segments']
        if not s:
            return {}
        return {
            'total'    : len(s),
            'critical' : sum(1 for x in s if x['severity'] == 'CRITICAL'),
            'warning'  : sum(1 for x in s if x['severity'] == 'WARNING'),
            'avg_rough': round(sum(x['roughness'] for x in s) / len(s), 1),
        }


df  = pd.read_csv('smart_ride_ai/data/suspension_data.csv')
db  = RoadMemoryDB()

for _, row in df.iterrows():
    db.add(row['ts'], row['roughness'], row['pothole_count'], row['pred_comfort'])

st = db.stats()
print('road memory saved')
for k, v in st.items():
    print(f'  {k}: {v}')


In [ ]:
# CELL 16 — predictive alert engine
import pandas as pd

RULES = [
    (lambda r: r['roughness'] > 80,           'ROAD_DMG_CRITICAL',   'CRITICAL'),
    (lambda r: r['roughness'] > 55,           'ROUGH_ROAD_AHEAD',    'WARNING'),
    (lambda r: r['pothole_count'] > 2,        'POTHOLE_MULTI',       'CRITICAL'),
    (lambda r: r['pothole_count'] > 0,        'POTHOLE_ZONE',        'WARNING'),
    (lambda r: r['rms'] > 2.0,                'VIBRATION_HIGH',      'HIGH'),
    (lambda r: r['pred_comfort'] < 25,        'COMFORT_CRITICAL',    'CRITICAL'),
    (lambda r: r['pred_comfort'] < 50,        'COMFORT_DROPPING',    'WARNING'),
    (lambda r: r['mode'] == 'ADAPTIVE_ROUGH', 'SUSPENSION_ADAPTING', 'INFO'),
]

LOOKAHEAD = 15

df     = pd.read_csv('smart_ride_ai/data/suspension_data.csv')
alerts = []

for i in range(len(df)):
    window = df.iloc[i: i + LOOKAHEAD]
    for fn, msg, sev in RULES:
        if window.apply(fn, axis=1).any():
            alerts.append({
                'frame': int(df.iloc[i]['frame']),
                'ts'   : float(df.iloc[i]['ts']),
                'alert': msg,
                'sev'  : sev,
            })
            break

df_alerts = pd.DataFrame(alerts)
df_alerts.to_csv('smart_ride_ai/data/alerts.csv', index=False)
print(f'{len(df_alerts)} alerts generated')
print(df_alerts['sev'].value_counts().to_string())
print()
print(df_alerts[['ts', 'alert', 'sev']].head(10).to_string())


---
## Phase 7 — Heatmap, analytics, Streamlit dashboard

In [ ]:
# CELL 17 — road heatmap + 9-panel analytics grid
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

df = pd.read_csv('smart_ride_ai/data/suspension_data.csv')

# heatmap
fig1, ax1 = plt.subplots(figsize=(13, 3))
fig1.patch.set_facecolor('#0a0a0f')
ax1.set_facecolor('#0a0a0f')

grid = np.zeros((5, len(df)))
for i, row in df.iterrows():
    lane = min(4, int(row['pothole_count']))
    grid[lane, i] = row['roughness']
    for d in [-1, 1]:
        if 0 <= lane + d < 5:
            grid[lane + d, i] = row['roughness'] * 0.5

cmap = LinearSegmentedColormap.from_list(
    'rh', ['#050510', '#003311', '#00ff44', '#ffdd00', '#ff4400', '#ff0000'], N=256)
im = ax1.imshow(grid, aspect='auto', cmap=cmap, vmin=0, vmax=100, interpolation='gaussian')
ax1.set_yticks(range(5))
ax1.set_yticklabels([f'lane {i}' for i in range(5)], color='#aaa', fontsize=8)
ax1.set_xlabel('frame', color='#aaa')
ax1.set_title('road roughness heatmap', color='#00ffcc', fontsize=11)
plt.colorbar(im, ax=ax1, label='roughness', fraction=0.015)
ax1.tick_params(colors='#555')
plt.tight_layout()
plt.savefig('smart_ride_ai/assets/heatmap.png', dpi=140, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

# 9-panel analytics
fig2 = plt.figure(figsize=(15, 11))
fig2.patch.set_facecolor('#0a0a0f')
gs = gridspec.GridSpec(3, 3, figure=fig2, hspace=0.5, wspace=0.35)
t  = df['ts']

panels = [
    (0, 0, 'comfort',          t, df['pred_comfort'],  '#00ffcc'),
    (0, 1, 'vibration RMS',    t, df['rms'],            '#ff4466'),
    (0, 2, 'roughness',        t, df['roughness'],      '#ff9900'),
    (1, 0, 'stability',        t, df['stability'],      '#00aaff'),
    (1, 1, 'accel Z',          t, df['az'],             '#ffdd00'),
    (1, 2, 'adaptive comfort', t, df['adaptive_c'],     '#aa00ff'),
    (2, 0, 'damping Ns/m',     t, df['damping'],        '#ff6699'),
    (2, 1, 'comfort gain',     t, df['gain'],           '#00ff88'),
    (2, 2, 'pothole count',    t, df['pothole_count'],  '#ff2244'),
]

for r, c, title, x, y, clr in panels:
    ax = fig2.add_subplot(gs[r, c])
    ax.set_facecolor('#111118')
    ax.plot(x, y, color=clr, lw=0.8)
    ax.fill_between(x, y, alpha=0.1, color=clr)
    ax.set_title(title, color='#aaa', fontsize=8)
    ax.tick_params(colors='#444', labelsize=6)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#1a1a2e')

fig2.suptitle('Smart Ride AI — analytics overview', color='#00ffcc', fontsize=13)
plt.savefig('smart_ride_ai/assets/analytics_dashboard.png', dpi=140,
            bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('charts saved')


In [ ]:
# CELL 18 — write Streamlit dashboard
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json, os

st.set_page_config(page_title="Smart Ride AI", page_icon="🚗", layout="wide")

st.markdown("""
<style>
.stApp{background:#07070f;color:#c8c8e8}
.kpi{background:#0d0d1a;border:1px solid #2a2a4a;border-radius:10px;padding:14px;text-align:center}
.kv{font-size:2rem;font-weight:700}
.kl{font-size:0.7rem;color:#666;letter-spacing:1.5px;text-transform:uppercase}
.ac{background:#1a0505;border-left:3px solid #ff2222;padding:6px 10px;border-radius:3px;margin:3px 0;font-size:0.82rem}
.aw{background:#1a1005;border-left:3px solid #ff9900;padding:6px 10px;border-radius:3px;margin:3px 0;font-size:0.82rem}
.ai{background:#05050f;border-left:3px solid #00aaff;padding:6px 10px;border-radius:3px;margin:3px 0;font-size:0.82rem}
h1,h2,h3{color:#00ffcc !important}
</style>
""", unsafe_allow_html=True)

@st.cache_data
def load():
    b = "smart_ride_ai/data"
    df = pd.read_csv(f"{b}/suspension_data.csv")
    al = pd.read_csv(f"{b}/alerts.csv") if os.path.exists(f"{b}/alerts.csv") else pd.DataFrame()
    mem = {}
    if os.path.exists(f"{b}/road_memory.json"):
        with open(f"{b}/road_memory.json") as f:
            mem = json.load(f)
    return df, al, mem

df, df_al, mem = load()

st.sidebar.title("control panel")
tr = st.sidebar.slider("time window (s)", float(df.ts.min()), float(df.ts.max()),
                        (float(df.ts.min()), float(df.ts.max())))
show_s = st.sidebar.checkbox("suspension comparison", True)
show_h = st.sidebar.checkbox("road heatmap", True)
show_m = st.sidebar.checkbox("road memory", True)

dv = df[(df.ts >= tr[0]) & (df.ts <= tr[1])]

st.title("Smart Ride AI")
st.caption("road condition · vibration · adaptive suspension · predictive alerts")
st.divider()

c1, c2, c3, c4, c5 = st.columns(5)

def kpi(col, val, label, good_thresh):
    try:
        numeric = float(str(val).replace("+", ""))
        clr = "#00ff88" if numeric > good_thresh else "#ff2222"
    except ValueError:
        clr = "#aaaaaa"
    col.markdown(f'<div class="kpi"><div class="kv" style="color:{clr}">{val}</div><div class="kl">{label}</div></div>',
                 unsafe_allow_html=True)

kpi(c1, f"{dv.pred_comfort.mean():.0f}",   "avg comfort",   60)
kpi(c2, f"{dv.rms.mean():.2f}",            "vibration rms",  99)
kpi(c3, f"{int(dv.pothole_count.sum())}",  "total potholes", 99)
kpi(c4, f"+{dv.gain.mean():.1f}",          "comfort gain",   0)
kpi(c5, f"{int((dv.roughness > 80).sum())}", "critical zones", 99)

st.divider()
cl, cr = st.columns([2, 1])

with cl:
    st.subheader("ride intelligence timeline")
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        subplot_titles=("comfort score", "vibration RMS", "roughness"))
    fig.add_trace(go.Scatter(x=dv.ts, y=dv.pred_comfort, fill="tozeroy",
                             line_color="#00ffcc", fillcolor="rgba(0,255,204,0.08)"), row=1, col=1)
    fig.add_trace(go.Scatter(x=dv.ts, y=dv.rms, fill="tozeroy",
                             line_color="#ff4466", fillcolor="rgba(255,68,102,0.08)"), row=2, col=1)
    fig.add_trace(go.Scatter(x=dv.ts, y=dv.roughness, fill="tozeroy",
                             line_color="#ff9900", fillcolor="rgba(255,153,0,0.08)"), row=3, col=1)
    fig.update_layout(height=400, paper_bgcolor="#0a0a0f", plot_bgcolor="#111118",
                      font_color="#aaa", showlegend=False)
    fig.update_xaxes(gridcolor="#1a1a2e")
    fig.update_yaxes(gridcolor="#1a1a2e")
    st.plotly_chart(fig, use_container_width=True)

with cr:
    st.subheader("live alerts")
    if not df_al.empty:
        filtered = df_al[(df_al.ts >= tr[0]) & (df_al.ts <= tr[1])].tail(14)
        for _, row in filtered.iterrows():
            css = "ac" if row.sev == "CRITICAL" else "aw" if row.sev == "WARNING" else "ai"
            st.markdown(f'<div class="{css}"><small>{row.ts:.1f}s</small>  {row.alert}</div>',
                        unsafe_allow_html=True)
    else:
        st.info("no alerts in range")

st.divider()

if show_s:
    st.subheader("adaptive vs passive suspension")
    f2 = go.Figure()
    f2.add_trace(go.Scatter(x=dv.ts, y=dv.adaptive_c, name="adaptive",
                            line=dict(color="#00ffcc", width=2)))
    f2.add_trace(go.Scatter(x=dv.ts, y=dv.passive_c, name="passive",
                            line=dict(color="#ff4466", width=1.5, dash="dash")))
    f2.update_layout(height=280, paper_bgcolor="#0a0a0f", plot_bgcolor="#111118", font_color="#aaa")
    f2.update_xaxes(gridcolor="#1a1a2e", title="time s")
    f2.update_yaxes(gridcolor="#1a1a2e", title="comfort")
    st.plotly_chart(f2, use_container_width=True)

if show_h:
    st.subheader("road roughness heatmap")
    pv = dv.copy()
    pv["lane"] = (pv["pothole_count"] % 5).astype(int)
    f3 = px.density_heatmap(pv, x="ts", y="lane", z="roughness", nbinsx=60, nbinsy=5,
                             color_continuous_scale=["#050510", "#003311", "#00ff44",
                                                     "#ffdd00", "#ff4400", "#ff0000"])
    f3.update_layout(height=220, paper_bgcolor="#0a0a0f", font_color="#aaa")
    st.plotly_chart(f3, use_container_width=True)

if show_m and mem.get("segments"):
    st.subheader("road memory")
    segs = pd.DataFrame(mem["segments"])
    ca, cb = st.columns([2, 1])
    with ca:
        st.dataframe(segs[["id", "ts", "roughness", "potholes", "comfort", "severity"]], height=280)
    with cb:
        vc = segs["severity"].value_counts()
        f4 = px.pie(values=vc.values, names=vc.index, hole=0.4,
                    color_discrete_map={"CRITICAL": "#ff2222", "WARNING": "#ff9900", "CAUTION": "#ffdd00"})
        f4.update_layout(paper_bgcolor="#0a0a0f", font_color="#aaa", height=260)
        st.plotly_chart(f4, use_container_width=True)

st.divider()
st.caption("Smart Ride AI · OpenCV · scikit-learn · Streamlit · Plotly")
'''

with open('smart_ride_ai/dashboard/app.py', 'w') as f:
    f.write(dashboard_code)
print('dashboard/app.py written')


In [ ]:
# CELL 19 — launch Streamlit via localtunnel
import subprocess, time

subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)

sp = subprocess.Popen([
    'streamlit', 'run', 'smart_ride_ai/dashboard/app.py',
    '--server.port', '8501',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false',
])

time.sleep(4)

tp = subprocess.Popen(['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

time.sleep(3)
url = tp.stdout.readline().strip()
print(f'dashboard live at: {url}')
print('if it asks for your IP run:  !curl https://ipv4.icanhazip.com')


---
## Phase 8 — Final summary

In [ ]:
# CELL 20 — final summary
import pandas as pd

df = pd.read_csv('smart_ride_ai/data/suspension_data.csv')
al = pd.read_csv('smart_ride_ai/data/alerts.csv')

print('smart ride ai — run complete')
print(f'  frames analyzed     : {len(df)}')
print(f'  potholes detected   : {int(df.pothole_count.sum())}')
print(f'  avg vibration RMS   : {df.rms.mean():.3f} m/s2')
print(f'  avg comfort score   : {df.pred_comfort.mean():.1f} / 100')
print(f'  avg suspension gain : +{df.gain.mean():.1f} pts')
print(f'  alerts generated    : {len(al)}')


In [ ]:
# CELL 21 — zip and download
import shutil
from google.colab import files

shutil.make_archive('smart_ride_ai_project', 'zip', '.', 'smart_ride_ai')
files.download('smart_ride_ai_project.zip')
print('download started')
